[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrybrew/bi_data_science_handson_2026/blob/main/notebooks/session2_ml.ipynb)

# Session 2 — Machine Learning for Early Warning

Bank Indonesia–oriented synthetic regional inflation dataset.

**Direct dataset link used in this notebook:**  
`https://raw.githubusercontent.com/andrybrew/bi_data_science_handson_2026/main/data/bi_regional_inflation_synthetic.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

DATA_URL = "https://raw.githubusercontent.com/andrybrew/bi_data_science_handson_2026/main/data/bi_regional_inflation_synthetic.csv"
df = pd.read_csv(DATA_URL)
df["month"] = pd.to_datetime(df["month"])
df["high_inflation_pressure"] = (df["cpi_inflation_yoy"] >= 2.5).astype(int)
df["month_num"] = df["month"].dt.month
df["quarter"] = df["month"].dt.quarter

df.head()

In [ ]:
features = [
    "province", "food_price_index", "exchange_rate_change_pct",
    "qris_transactions_growth_pct", "online_sentiment_score",
    "unemployment_rate", "hotel_occupancy_rate", "rainfall_index",
    "month_num", "quarter"
]
target = "high_inflation_pressure"

X = df[features]
y = df[target]

categorical_features = ["province"]
numeric_features = [
    "food_price_index", "exchange_rate_change_pct",
    "qris_transactions_growth_pct", "online_sentiment_score",
    "unemployment_rate", "hotel_occupancy_rate", "rainfall_index",
    "month_num", "quarter"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y.value_counts())

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe.named_steps["model"], "predict_proba") else None

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob) if y_prob is not None else None
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
results_df

In [ ]:
best_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)
y_prob = best_pipe.predict_proba(X_test)[:, 1]

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix - Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve - Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
ohe = best_pipe.named_steps["prep"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(["province"])
feature_names = list(cat_names) + numeric_features

importances = best_pipe.named_steps["model"].feature_importances_
feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp.head(15)